# 🎓 Student Math Score Prediction Pipeline

A comprehensive machine learning pipeline for predicting student math scores using ensemble methods.

## Pipeline Overview

1. **Data Loading & Preprocessing** - Memory-optimized loading with categorical encoding
2. **Baseline Models** - LightGBM, XGBoost, CatBoost comparison
3. **Hyperparameter Optimization** - Optuna-based tuning
4. **Feature Selection** - Recursive feature elimination
5. **Advanced Ensembling** - Stacking with meta-learners
6. **Model Interpretability** - SHAP analysis

---

## 1. Environment Setup

In [ ]:
# Install required packages (run once)
!pip install -q lightgbm xgboost catboost optuna shap lime

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

# Core Libraries
import gc
import json
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

# Data Manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import shap

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.cluster import KMeans

# Gradient Boosting Models
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, CatBoostClassifier

# Hyperparameter Optimization
import optuna
from tqdm.auto import tqdm

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Model Persistence
import joblib

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Constants
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All imports successful!")

---
## 2. Data Loading & Preprocessing

In [ ]:
# ============================================================================
# DATA LOADING UTILITIES
# ============================================================================

def reduce_memory_usage(df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    """
    Reduce DataFrame memory usage by downcasting numeric types.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame to optimize
    verbose : bool
        Whether to print memory reduction stats
    
    Returns
    -------
    pd.DataFrame
        Memory-optimized DataFrame
    """
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and col_type.name != 'category':
            c_min, c_max = df[col].min(), df[col].max()
            
            if str(col_type).startswith('int'):
                # Downcast integers
                for dtype in [np.int8, np.int16, np.int32]:
                    if c_min > np.iinfo(dtype).min and c_max < np.iinfo(dtype).max:
                        df[col] = df[col].astype(dtype)
                        break
            else:
                # Downcast floats to float32
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
        
        elif col_type == object:
            df[col] = df[col].astype('category')
    
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    
    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem
        print(f"   Memory: {start_mem:.1f}MB → {end_mem:.1f}MB ({reduction:.1f}% reduction)")
    
    return df


def load_data(
    x_path: str,
    y_path: str,
    chunksize: int = 100_000,
    nrows: Optional[int] = None
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Load training data with memory optimization.
    
    Parameters
    ----------
    x_path : str
        Path to features CSV
    y_path : str
        Path to target CSV
    chunksize : int
        Rows per chunk for streaming large files
    nrows : int, optional
        Limit rows for quick testing
    
    Returns
    -------
    Tuple[pd.DataFrame, pd.Series]
        Features and target
    """
    print("📂 Loading Data...")
    
    # Load target
    y = pd.read_csv(y_path, low_memory=False, index_col=0, nrows=nrows)
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    
    # Load features in chunks (memory efficient)
    if nrows:
        X = pd.read_csv(x_path, low_memory=False, index_col=0, nrows=nrows)
        X = reduce_memory_usage(X, verbose=True)
    else:
        chunks = []
        for chunk in pd.read_csv(x_path, chunksize=chunksize, low_memory=False, index_col=0):
            chunks.append(reduce_memory_usage(chunk))
        X = pd.concat(chunks)
    
    # Align lengths
    min_len = min(len(X), len(y))
    X, y = X.iloc[:min_len], y.iloc[:min_len]
    
    print(f"   ✓ Loaded {len(X):,} samples with {X.shape[1]} features")
    
    return X, y

In [ ]:
# ============================================================================
# DATA PREPARATION
# ============================================================================

def prepare_datasets(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = 0.15,
    val_size: float = 0.1765
) -> Tuple[Tuple, Tuple, Dict]:
    """
    Split data and prepare for boosting and sklearn models.
    
    Creates two versions:
    - Boosting: Categorical codes for LightGBM/XGBoost/CatBoost
    - Sklearn: Imputed numeric data for traditional ML
    
    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target variable
    test_size : float
        Proportion for test set
    val_size : float
        Proportion of train for validation
    
    Returns
    -------
    Tuple containing:
        - data_boost: (X_train, y_train, X_val, y_val, X_test, y_test)
        - data_sklearn: Same structure with imputed data
        - category_map: Mapping of categorical encodings
    """
    print("🔀 Splitting Data...")
    
    # Train/Test split
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE
    )
    
    # Train/Validation split
    X_train_raw, X_val_raw, y_train, y_val = train_test_split(
        X_train_raw, y_train, test_size=val_size, random_state=RANDOM_STATE
    )
    
    del X
    gc.collect()
    
    print(f"   Train: {len(X_train_raw):,} | Val: {len(X_val_raw):,} | Test: {len(X_test_raw):,}")
    
    # === Prepare Boosting Dataset ===
    print("⚙️  Encoding categorical features...")
    
    X_train_boost = X_train_raw.copy()
    X_val_boost = X_val_raw.copy()
    X_test_boost = X_test_raw.copy()
    
    del X_train_raw, X_val_raw, X_test_raw
    gc.collect()
    
    cat_cols = X_train_boost.select_dtypes(include=['category', 'object']).columns
    category_map = {}
    
    for col in cat_cols:
        # Define categories from training data only
        train_categories = X_train_boost[col].astype('category').cat.categories
        category_map[col] = train_categories
        
        strict_dtype = pd.CategoricalDtype(categories=train_categories, ordered=True)
        
        # Apply consistent encoding (unseen categories become -1)
        X_train_boost[col] = X_train_boost[col].astype(strict_dtype).cat.codes
        X_val_boost[col] = X_val_boost[col].astype(strict_dtype).cat.codes
        X_test_boost[col] = X_test_boost[col].astype(strict_dtype).cat.codes
    
    # === Prepare Sklearn Dataset (Imputed) ===
    print("⚙️  Preparing sklearn-compatible data...")
    
    medians = X_train_boost.median(numeric_only=True)
    empty_cols = medians[medians.isna()].index
    
    if len(empty_cols) > 0:
        medians = medians.drop(empty_cols)
    
    X_train_sklearn = X_train_boost.drop(columns=empty_cols).fillna(medians).astype(np.float32)
    X_val_sklearn = X_val_boost.drop(columns=empty_cols).fillna(medians).astype(np.float32)
    X_test_sklearn = X_test_boost.drop(columns=empty_cols).fillna(medians).astype(np.float32)
    
    print("   ✓ Data preparation complete!")
    
    data_boost = (X_train_boost, y_train, X_val_boost, y_val, X_test_boost, y_test)
    data_sklearn = (X_train_sklearn, y_train, X_val_sklearn, y_val, X_test_sklearn, y_test)
    
    return data_boost, data_sklearn, category_map

In [ ]:
# ============================================================================
# TEST SET PREPARATION
# ============================================================================

def prepare_test_data(
    test_path: str,
    train_columns: pd.Index,
    category_map: Dict,
    nrows: Optional[int] = None
) -> pd.DataFrame:
    """
    Prepare official test set to match training data structure.
    
    Parameters
    ----------
    test_path : str
        Path to test CSV
    train_columns : pd.Index
        Column order from training data
    category_map : Dict
        Categorical encodings from training
    nrows : int, optional
        Limit rows for testing
    
    Returns
    -------
    pd.DataFrame
        Processed test features
    """
    print("📂 Processing Test Set...")
    
    if nrows:
        X_test = pd.read_csv(test_path, nrows=nrows, low_memory=False, index_col=0)
        X_test = reduce_memory_usage(X_test)
    else:
        chunks = []
        for chunk in pd.read_csv(test_path, chunksize=100_000, low_memory=False, index_col=0):
            chunks.append(reduce_memory_usage(chunk))
        X_test = pd.concat(chunks)
    
    # Align columns with training data
    X_test = X_test.reindex(columns=train_columns)
    
    # Apply categorical encodings
    for col, categories in category_map.items():
        if col in X_test.columns:
            strict_dtype = pd.CategoricalDtype(categories=categories, ordered=True)
            X_test[col] = X_test[col].astype(strict_dtype).cat.codes
    
    print(f"   ✓ Processed {len(X_test):,} test samples")
    
    return X_test


def align_features_by_test_quality(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    missing_threshold: float = 0.90
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Remove features that are mostly missing in test set (potential leakage).
    
    Parameters
    ----------
    X_train, X_val, X_test : pd.DataFrame
        Feature matrices
    missing_threshold : float
        Drop features with missingness above this threshold
    
    Returns
    -------
    Tuple of cleaned DataFrames and list of dropped columns
    """
    print(f"🔍 Filtering features (>{missing_threshold:.0%} missing in test)...")
    
    missing_ratio = X_test.isna().mean()
    cols_to_drop = missing_ratio[missing_ratio > missing_threshold].index.tolist()
    
    print(f"   → Dropping {len(cols_to_drop)} features")
    if cols_to_drop:
        print(f"   Examples: {cols_to_drop[:5]}")
    
    X_train_clean = X_train.drop(columns=cols_to_drop, errors='ignore')
    X_val_clean = X_val.drop(columns=cols_to_drop, errors='ignore')
    X_test_clean = X_test.drop(columns=cols_to_drop, errors='ignore')
    
    return X_train_clean, X_val_clean, X_test_clean, cols_to_drop

---
## 3. Baseline Model Comparison

In [ ]:
# ============================================================================
# MODEL TRAINING & EVALUATION
# ============================================================================

def train_baseline_models(
    data_boost: Tuple,
    verbose: bool = True
) -> Tuple[Dict[str, float], Dict[str, Any], np.ndarray]:
    """
    Train and evaluate baseline gradient boosting models.
    
    Parameters
    ----------
    data_boost : Tuple
        (X_train, y_train, X_val, y_val, X_test, y_test)
    verbose : bool
        Print progress
    
    Returns
    -------
    Tuple containing:
        - results: Dict of model RMSE scores
        - models: Dict of trained models
        - predictions: Dict of test predictions
    """
    X_train, y_train, X_val, y_val, X_test, y_test = data_boost
    
    results = {}
    models = {}
    predictions = {}
    
    print("\n" + "=" * 50)
    print("🚀 BASELINE MODEL EVALUATION")
    print("=" * 50)
    
    # --- LightGBM ---
    print("\n📊 1. LightGBM")
    lgbm = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=-1
    )
    lgbm.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    pred_lgbm = lgbm.predict(X_test)
    rmse_lgbm = np.sqrt(mean_squared_error(y_test, pred_lgbm))
    
    results['LightGBM'] = rmse_lgbm
    models['LightGBM'] = lgbm
    predictions['LightGBM'] = pred_lgbm
    print(f"   Test RMSE: {rmse_lgbm:.4f}")
    
    # --- XGBoost ---
    print("\n📊 2. XGBoost")
    xgbr = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=8,
        tree_method='hist',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        enable_categorical=True,
        early_stopping_rounds=50
    )
    xgbr.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred_xgb = xgbr.predict(X_test)
    rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
    
    results['XGBoost'] = rmse_xgb
    models['XGBoost'] = xgbr
    predictions['XGBoost'] = pred_xgb
    print(f"   Test RMSE: {rmse_xgb:.4f}")
    
    # --- CatBoost ---
    print("\n📊 3. CatBoost")
    catb = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        allow_writing_files=False,
        verbose=False,
        random_state=RANDOM_STATE
    )
    catb.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)
    pred_cat = catb.predict(X_test)
    rmse_cat = np.sqrt(mean_squared_error(y_test, pred_cat))
    
    results['CatBoost'] = rmse_cat
    models['CatBoost'] = catb
    predictions['CatBoost'] = pred_cat
    print(f"   Test RMSE: {rmse_cat:.4f}")
    
    # --- Simple Average Ensemble ---
    print("\n📊 4. Voting Ensemble (Average)")
    pred_vote = (pred_lgbm + pred_xgb + pred_cat) / 3
    rmse_vote = np.sqrt(mean_squared_error(y_test, pred_vote))
    
    results['Voting Ensemble'] = rmse_vote
    predictions['Voting'] = pred_vote
    print(f"   Test RMSE: {rmse_vote:.4f}")
    
    # Summary
    print("\n" + "=" * 50)
    print("📈 RESULTS SUMMARY")
    print("=" * 50)
    for name, rmse in sorted(results.items(), key=lambda x: x[1]):
        print(f"   {name:20s}: {rmse:.4f}")
    
    return results, models, predictions

---
## 4. Hyperparameter Optimization

In [ ]:
# ============================================================================
# OPTUNA HYPERPARAMETER TUNING
# ============================================================================

class CatBoostOptimizer:
    """
    Optuna-based hyperparameter optimizer for CatBoost.
    """
    
    def __init__(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        use_gpu: bool = False
    ):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.use_gpu = use_gpu
    
    def objective(self, trial: optuna.Trial) -> float:
        """Optuna objective function."""
        
        params = {
            'iterations': trial.suggest_int('iterations', 5000, 10000),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'random_strength': trial.suggest_int('random_strength', 0, 100),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'loss_function': 'RMSE',
            'verbose': False,
            'random_state': RANDOM_STATE,
            'allow_writing_files': False
        }
        
        if self.use_gpu:
            params['task_type'] = 'GPU'
            params['devices'] = '0'
        
        try:
            model = CatBoostRegressor(**params)
            model.fit(
                self.X_train, self.y_train,
                eval_set=(self.X_val, self.y_val),
                early_stopping_rounds=50,
                verbose=False
            )
        except Exception as e:
            # Fallback to CPU if GPU fails
            if self.use_gpu:
                params.pop('task_type', None)
                params.pop('devices', None)
                model = CatBoostRegressor(**params)
                model.fit(
                    self.X_train, self.y_train,
                    eval_set=(self.X_val, self.y_val),
                    early_stopping_rounds=50,
                    verbose=False
                )
            else:
                raise e
        
        preds = model.predict(self.X_val)
        rmse = np.sqrt(mean_squared_error(self.y_val, preds))
        
        return rmse
    
    def optimize(self, n_trials: int = 50) -> Dict:
        """Run optimization study."""
        
        study = optuna.create_study(direction='minimize')
        
        print(f"🔧 Starting hyperparameter optimization ({n_trials} trials)...")
        
        with tqdm(total=n_trials, desc="Optimizing") as pbar:
            for i in range(n_trials):
                study.optimize(self.objective, n_trials=1, show_progress_bar=False)
                pbar.update(1)
                pbar.set_postfix({'Best RMSE': f"{study.best_value:.4f}"})
        
        print(f"\n✅ Best RMSE: {study.best_value:.4f}")
        print(f"   Best params: {study.best_params}")
        
        return study.best_params


def save_params(params: Dict, filepath: str) -> None:
    """Save parameters to JSON file."""
    with open(filepath, 'w') as f:
        json.dump(params, f, indent=4)
    print(f"💾 Saved parameters to {filepath}")


def load_params(filepath: str) -> Dict:
    """Load parameters from JSON file."""
    with open(filepath, 'r') as f:
        params = json.load(f)
    print(f"📂 Loaded parameters from {filepath}")
    return params

---
## 5. Feature Selection

In [ ]:
# ============================================================================
# RECURSIVE FEATURE ELIMINATION
# ============================================================================

def recursive_feature_selection(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    params: Dict,
    n_steps: int = 10
) -> Tuple[List[str], Dict[int, float]]:
    """
    Select optimal features using recursive elimination based on importance.
    
    Parameters
    ----------
    X_train, y_train : Training data
    X_val, y_val : Validation data
    params : Model parameters
    n_steps : Number of feature count levels to test
    
    Returns
    -------
    Tuple of (selected_features, results_dict)
    """
    print("🔍 Starting Recursive Feature Selection...")
    
    # Reduce iterations for speed
    selection_params = params.copy()
    selection_params['iterations'] = 1000
    selection_params['verbose'] = False
    selection_params['allow_writing_files'] = False
    
    # Initial training to get feature importance
    print("   Training baseline model...")
    full_model = CatBoostRegressor(**selection_params)
    full_model.fit(X_train, y_train)
    
    # Rank features by importance
    importance = full_model.get_feature_importance()
    sorted_idx = np.argsort(importance)[::-1]
    sorted_features = X_train.columns[sorted_idx]
    
    # Define feature counts to test
    total_features = len(sorted_features)
    steps = [int(x) for x in np.linspace(total_features, 10, n_steps)]
    
    results = {}
    
    print("   Testing feature subsets...")
    for n in tqdm(steps, desc="Feature Selection"):
        top_features = sorted_features[:n]
        
        model = CatBoostRegressor(**selection_params)
        model.fit(X_train[top_features], y_train)
        
        preds = model.predict(X_val[top_features])
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        
        results[n] = rmse
    
    # Find optimal
    best_n = min(results, key=results.get)
    best_features = list(sorted_features[:best_n])
    
    # Plot elbow curve
    plt.figure(figsize=(10, 6))
    plt.plot(list(results.keys()), list(results.values()), 'bo-', linewidth=2, markersize=8)
    plt.axvline(x=best_n, color='r', linestyle='--', label=f'Optimal: {best_n} features')
    plt.xlabel('Number of Features', fontsize=12)
    plt.ylabel('Validation RMSE', fontsize=12)
    plt.title('Feature Selection: RMSE vs Number of Features', fontsize=14)
    plt.gca().invert_xaxis()
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Optimal: {best_n} features (RMSE: {results[best_n]:.4f})")
    
    return best_features, results

---
## 6. Advanced Ensembling

In [ ]:
# ============================================================================
# STACKING ENSEMBLE
# ============================================================================

def build_stacking_ensemble(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    catboost_params: Dict,
    cv_folds: int = 5,
    use_gpu: bool = False
) -> StackingRegressor:
    """
    Build a stacking ensemble with CatBoost and XGBoost base learners.
    
    Parameters
    ----------
    X_train, y_train : Training data
    catboost_params : Optimized CatBoost parameters
    cv_folds : Cross-validation folds for stacking
    use_gpu : Whether to use GPU acceleration
    
    Returns
    -------
    StackingRegressor
        Trained stacking ensemble
    """
    print("🏗️  Building Stacking Ensemble...")
    
    # Prepare CatBoost parameters
    cat_params = catboost_params.copy()
    cat_params['verbose'] = 0
    cat_params['allow_writing_files'] = False
    
    if use_gpu:
        cat_params['task_type'] = 'GPU'
        cat_params['devices'] = '0'
    
    # Define base estimators
    estimators = [
        ('catboost', CatBoostRegressor(**cat_params)),
        ('xgboost', xgb.XGBRegressor(
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=6,
            tree_method='hist',
            device='cuda' if use_gpu else 'cpu',
            random_state=RANDOM_STATE,
            verbosity=0
        ))
    ]
    
    # Meta-learner with automatic regularization
    meta_learner = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0])
    
    # Build stacking model
    stacker = StackingRegressor(
        estimators=estimators,
        final_estimator=meta_learner,
        cv=cv_folds,
        n_jobs=1,
        passthrough=False
    )
    
    print("   Training stacking ensemble (this may take a while)...")
    stacker.fit(X_train, y_train)
    
    print("   ✓ Stacking ensemble trained!")
    
    return stacker


def build_meta_ensemble(
    predictions_dict: Dict[str, np.ndarray],
    y_val: pd.Series
) -> Tuple[RidgeCV, pd.Series]:
    """
    Build a meta-learner to blend multiple model predictions.
    
    Parameters
    ----------
    predictions_dict : Dict[str, np.ndarray]
        Dictionary of model_name -> validation predictions
    y_val : pd.Series
        True validation targets
    
    Returns
    -------
    Tuple of (meta_model, learned_weights)
    """
    print("🔗 Building Meta-Ensemble...")
    
    # Create meta-features from predictions
    X_meta = pd.DataFrame(predictions_dict)
    
    # Train Ridge meta-learner
    meta_model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0])
    meta_model.fit(X_meta, y_val)
    
    # Extract weights
    weights = pd.Series(meta_model.coef_, index=X_meta.columns)
    
    print(f"   Optimal alpha: {meta_model.alpha_}")
    print("   Learned weights:")
    for name, weight in weights.sort_values(ascending=False).items():
        print(f"      {name}: {weight:.4f}")
    
    # Evaluate
    meta_preds = meta_model.predict(X_meta)
    r2 = r2_score(y_val, meta_preds)
    print(f"\n   ✓ Meta-Ensemble Validation R²: {r2:.5f}")
    
    return meta_model, weights

---
## 7. Advanced Techniques

In [ ]:
# ============================================================================
# ADVERSARIAL VALIDATION
# ============================================================================

def adversarial_validation(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    use_gpu: bool = False
) -> Tuple[float, Optional[pd.DataFrame]]:
    """
    Check for distribution shift between train and test sets.
    
    If AUC > 0.55, there's significant shift. Returns filtered
    training data that looks more like test data.
    
    Parameters
    ----------
    X_train, X_test : Feature matrices
    use_gpu : Whether to use GPU
    
    Returns
    -------
    Tuple of (auc_score, filtered_train_mask)
    """
    print("🔬 Running Adversarial Validation...")
    
    # Create synthetic target
    X_adv_train = X_train.copy()
    X_adv_test = X_test.copy()
    
    X_adv_train['is_test'] = 0
    X_adv_test['is_test'] = 1
    
    X_adv = pd.concat([X_adv_train, X_adv_test], axis=0)
    y_adv = X_adv['is_test']
    X_adv = X_adv.drop(columns=['is_test'])
    
    # Cross-validated predictions
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(X_adv))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_adv, y_adv)):
        X_tr, X_vl = X_adv.iloc[train_idx], X_adv.iloc[val_idx]
        y_tr, y_vl = y_adv.iloc[train_idx], y_adv.iloc[val_idx]
        
        clf_params = {
            'iterations': 500,
            'eval_metric': 'AUC',
            'verbose': 0,
            'allow_writing_files': False
        }
        if use_gpu:
            clf_params['task_type'] = 'GPU'
            clf_params['devices'] = '0'
        
        clf = CatBoostClassifier(**clf_params)
        clf.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), early_stopping_rounds=50)
        
        oof_preds[val_idx] = clf.predict_proba(X_vl)[:, 1]
    
    auc = roc_auc_score(y_adv, oof_preds)
    print(f"   Adversarial AUC: {auc:.4f}")
    
    if auc < 0.55:
        print("   ✓ Distributions match - no filtering needed")
        return auc, None
    else:
        print("   ⚠️  Distribution shift detected - filtering training data")
        
        # Keep training samples that look like test
        train_probs = oof_preds[:len(X_train)]
        threshold = np.percentile(train_probs, 30)
        mask = train_probs > threshold
        
        print(f"   Keeping {mask.sum():,} of {len(mask):,} training samples")
        
        return auc, mask

In [ ]:
# ============================================================================
# CLUSTER-BASED MODELING
# ============================================================================

def cluster_based_modeling(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    params: Dict,
    n_clusters: int = 6
) -> Tuple[np.ndarray, Dict]:
    """
    Train separate models for different student clusters.
    
    Parameters
    ----------
    X_train, y_train : Training data
    X_val, y_val : Validation data  
    X_test : Test features
    params : CatBoost parameters
    n_clusters : Number of student clusters
    
    Returns
    -------
    Tuple of (test_predictions, cluster_models)
    """
    print(f"👥 Training Cluster-Based Models ({n_clusters} clusters)...")
    
    # Impute for clustering
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_val_imp = imputer.transform(X_val)
    X_test_imp = imputer.transform(X_test)
    
    # Scale for K-Means
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    
    # Cluster
    print("   Running K-Means clustering...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
    train_clusters = kmeans.fit_predict(X_train_scaled)
    val_clusters = kmeans.predict(X_val_scaled)
    test_clusters = kmeans.predict(X_test_scaled)
    
    print(f"   Cluster sizes: {np.bincount(train_clusters)}")
    
    # Train cluster-specific models
    cluster_models = {}
    model_params = params.copy()
    model_params['verbose'] = 0
    model_params['allow_writing_files'] = False
    
    for k in range(n_clusters):
        mask_train = train_clusters == k
        mask_val = val_clusters == k
        
        X_tr_k = X_train[mask_train]
        y_tr_k = y_train[mask_train]
        X_val_k = X_val[mask_val]
        y_val_k = y_val[mask_val]
        
        model = CatBoostRegressor(**model_params)
        model.fit(X_tr_k, y_tr_k, eval_set=[(X_val_k, y_val_k)] if len(X_val_k) > 0 else None)
        
        cluster_models[k] = model
        
        if len(X_val_k) > 0:
            rmse = model.best_score_.get('validation', {}).get('RMSE', 'N/A')
            print(f"   Cluster {k}: {mask_train.sum()} samples, Val RMSE: {rmse}")
    
    # Predict on test set
    test_predictions = np.zeros(len(X_test))
    for k in range(n_clusters):
        mask = test_clusters == k
        if mask.sum() > 0:
            test_predictions[mask] = cluster_models[k].predict(X_test[mask])
    
    print("   ✓ Cluster-based predictions complete!")
    
    return test_predictions, cluster_models

---
## 8. Neural Network Error Correction

In [ ]:
# ============================================================================
# NEURAL NETWORK ERROR CORRECTOR
# ============================================================================

class ErrorCorrectorMLP(nn.Module):
    """
    MLP that predicts residual errors from base model predictions.
    """
    
    def __init__(self, input_dim: int):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            
            nn.Linear(128, 1)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


class ErrorCorrectionTrainer:
    """
    Trains a neural network to correct base model errors.
    """
    
    def __init__(self, device: str = 'auto'):
        if device == 'auto':
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)
        
        self.imputer = None
        self.scaler_X = None
        self.scaler_y = None
        self.model = None
    
    def _prepare_data(
        self,
        X: pd.DataFrame,
        y: Optional[pd.Series] = None,
        base_preds: Optional[np.ndarray] = None,
        is_train: bool = True
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Prepare data for neural network."""
        
        # Impute missing values
        if is_train:
            self.imputer = SimpleImputer(strategy='median')
            X_imp = self.imputer.fit_transform(X)
        else:
            X_imp = self.imputer.transform(X)
        
        # Scale features
        if is_train:
            self.scaler_X = StandardScaler()
            X_scaled = self.scaler_X.fit_transform(X_imp)
        else:
            X_scaled = self.scaler_X.transform(X_imp)
        
        X_tensor = torch.FloatTensor(X_scaled)
        y_tensor = None
        
        if is_train and y is not None and base_preds is not None:
            # Calculate residuals
            y_arr = y.values if hasattr(y, 'values') else y
            residuals = (y_arr.flatten() - base_preds.flatten()).reshape(-1, 1)
            
            # Scale residuals
            self.scaler_y = StandardScaler()
            y_scaled = self.scaler_y.fit_transform(residuals)
            y_tensor = torch.FloatTensor(y_scaled)
        
        return X_tensor, y_tensor
    
    def train(
        self,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        base_preds: np.ndarray,
        epochs: int = 100,
        batch_size: int = 2048,
        lr: float = 1e-3
    ) -> None:
        """Train the error corrector."""
        
        print(f"🧠 Training Error Corrector on {self.device}...")
        
        X_tensor, y_tensor = self._prepare_data(X_val, y_val, base_preds, is_train=True)
        
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        self.model = ErrorCorrectorMLP(X_tensor.shape[1]).to(self.device)
        optimizer = optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.MSELoss()
        
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for batch_X, batch_y in loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)
                
                optimizer.zero_grad()
                output = self.model(batch_X)
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                avg_loss = total_loss / len(loader)
                print(f"   Epoch {epoch+1:3d} | Loss: {avg_loss:.5f}")
        
        print("   ✓ Error corrector trained!")
    
    def predict(self, X: pd.DataFrame, base_preds: np.ndarray) -> np.ndarray:
        """Predict corrections and apply to base predictions."""
        
        X_tensor, _ = self._prepare_data(X, is_train=False)
        X_tensor = X_tensor.to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            corrections_scaled = self.model(X_tensor).cpu().numpy()
            corrections = self.scaler_y.inverse_transform(corrections_scaled).flatten()
        
        return base_preds + corrections

---
## 9. Model Interpretability (SHAP)

In [ ]:
# ============================================================================
# SHAP ANALYSIS
# ============================================================================

def explain_model(
    model: CatBoostRegressor,
    X: pd.DataFrame,
    sample_size: int = 5000
) -> shap.Explanation:
    """
    Generate SHAP explanations for model predictions.
    
    Parameters
    ----------
    model : Trained CatBoostRegressor
    X : Feature matrix
    sample_size : Number of samples for SHAP calculation
    
    Returns
    -------
    shap.Explanation object
    """
    print("🔍 Calculating SHAP values...")
    
    # Sample if dataset is large
    if len(X) > sample_size:
        X_sample = X.sample(sample_size, random_state=RANDOM_STATE)
    else:
        X_sample = X
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    print(f"   ✓ SHAP values calculated for {len(X_sample)} samples")
    
    return shap_values, X_sample, explainer


def plot_shap_summary(shap_values: np.ndarray, X: pd.DataFrame) -> None:
    """Plot SHAP summary (feature importance)."""
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X, show=False)
    plt.title("SHAP Summary: Feature Importance", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_shap_waterfall(
    shap_values: np.ndarray,
    X: pd.DataFrame,
    explainer: shap.TreeExplainer,
    sample_idx: int = 0,
    max_display: int = 10
) -> None:
    """Plot SHAP waterfall for individual prediction."""
    
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values[sample_idx],
            base_values=explainer.expected_value,
            data=X.iloc[sample_idx],
            feature_names=X.columns.tolist()
        ),
        max_display=max_display
    )
    plt.tight_layout()


def plot_shap_dependence(
    shap_values: np.ndarray,
    X: pd.DataFrame,
    feature: str,
    interaction_feature: Optional[str] = None
) -> None:
    """Plot SHAP dependence for a feature."""
    
    shap.dependence_plot(
        feature,
        shap_values,
        X,
        interaction_index=interaction_feature
    )

---
## 10. Utilities

In [ ]:
# ============================================================================
# MODEL PERSISTENCE & SUBMISSION
# ============================================================================

def save_model(model: Any, filepath: str) -> None:
    """Save model to disk."""
    
    if isinstance(model, CatBoostRegressor):
        model.save_model(filepath, format="cbm")
    else:
        joblib.dump(model, filepath)
    
    print(f"💾 Model saved to {filepath}")


def load_model(filepath: str, model_type: str = 'auto') -> Any:
    """Load model from disk."""
    
    if filepath.endswith('.cbm') or model_type == 'catboost':
        model = CatBoostRegressor()
        model.load_model(filepath)
    else:
        model = joblib.load(filepath)
    
    print(f"📂 Model loaded from {filepath}")
    return model


def create_submission(
    predictions: np.ndarray,
    test_index: pd.Index,
    y_train: pd.Series,
    filepath: str,
    clip_to_train_range: bool = True
) -> pd.DataFrame:
    """
    Create submission file with optional clipping.
    
    Parameters
    ----------
    predictions : Model predictions
    test_index : Index for test samples
    y_train : Training targets (for clipping range)
    filepath : Output path
    clip_to_train_range : Whether to clip predictions
    
    Returns
    -------
    pd.DataFrame
        Submission DataFrame
    """
    submission = pd.DataFrame({
        'ID': test_index,
        'MathScore': predictions
    })
    
    if clip_to_train_range:
        min_val, max_val = y_train.min(), y_train.max()
        submission['MathScore'] = submission['MathScore'].clip(min_val, max_val)
        print(f"   Clipped predictions to [{min_val:.2f}, {max_val:.2f}]")
    
    submission.to_csv(filepath, index=False)
    print(f"📄 Submission saved to {filepath}")
    
    return submission


def evaluate_predictions(
    y_true: pd.Series,
    y_pred: np.ndarray,
    name: str = "Model"
) -> Dict[str, float]:
    """Calculate and display evaluation metrics."""
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n📊 {name} Evaluation:")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   R²:   {r2:.4f}")
    
    return {'rmse': rmse, 'r2': r2}

---
## 11. Main Execution Pipeline

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# File paths
X_TRAIN_PATH = 'X_train.csv'
Y_TRAIN_PATH = 'y_train.csv'
X_TEST_PATH = 'X_test.csv'

# Model settings
USE_GPU = True
QUICK_MODE = False  # Set True for quick testing with subset of data
NROWS = 1000 if QUICK_MODE else None

print("⚙️  Configuration:")
print(f"   GPU: {USE_GPU}")
print(f"   Quick Mode: {QUICK_MODE}")

In [ ]:
# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================

X, y = load_data(X_TRAIN_PATH, Y_TRAIN_PATH, nrows=NROWS)

# Prepare datasets
data_boost, data_sklearn, category_map = prepare_datasets(X, y)

# Unpack
X_train, y_train, X_val, y_val, X_test_split, y_test = data_boost

# Clean up
del X, y
gc.collect()

In [ ]:
# ============================================================================
# STEP 2: PREPARE OFFICIAL TEST SET
# ============================================================================

X_official_test = prepare_test_data(X_TEST_PATH, X_train.columns, category_map, nrows=NROWS)

# Align features (remove leaky columns)
X_train_clean, X_val_clean, X_test_clean, dropped_cols = align_features_by_test_quality(
    X_train, X_val, X_official_test
)

In [ ]:
# ============================================================================
# STEP 3: BASELINE MODEL COMPARISON (Optional)
# ============================================================================

# Uncomment to run baseline comparison
# results, baseline_models, predictions = train_baseline_models(data_boost)

In [ ]:
# ============================================================================
# STEP 4: HYPERPARAMETER OPTIMIZATION (Optional - Time Consuming)
# ============================================================================

# Uncomment to run optimization
# optimizer = CatBoostOptimizer(X_train_clean, y_train, X_val_clean, y_val, use_gpu=USE_GPU)
# best_params = optimizer.optimize(n_trials=50)
# save_params(best_params, 'best_hyperparameters.json')

# Or load pre-optimized parameters
best_params = {
    'iterations': 7888,
    'depth': 6,
    'learning_rate': 0.0299,
    'random_strength': 29,
    'bagging_temperature': 0.896,
    'l2_leaf_reg': 2.29
}

print("📋 Using parameters:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

In [ ]:
# ============================================================================
# STEP 5: FEATURE SELECTION (Optional)
# ============================================================================

# Uncomment to run feature selection
# selected_features, selection_results = recursive_feature_selection(
#     X_train_clean, y_train, X_val_clean, y_val, best_params
# )

# For now, use all features
selected_features = X_train_clean.columns.tolist()
print(f"📊 Using {len(selected_features)} features")

In [ ]:
# ============================================================================
# STEP 6: TRAIN FINAL MODEL
# ============================================================================

# Prepare final parameters
final_params = best_params.copy()
final_params['iterations'] = 10000
final_params['early_stopping_rounds'] = 100
final_params['verbose'] = 500
final_params['allow_writing_files'] = False
final_params['random_state'] = RANDOM_STATE

if USE_GPU:
    final_params['task_type'] = 'GPU'
    final_params['devices'] = '0'

# Select features
X_train_sel = X_train_clean[selected_features]
X_val_sel = X_val_clean[selected_features]
X_test_sel = X_test_clean[selected_features]

# Train
print("\n🚀 Training Final CatBoost Model...")
final_model = CatBoostRegressor(**final_params)
final_model.fit(
    X_train_sel, y_train,
    eval_set=[(X_val_sel, y_val)],
    use_best_model=True
)

print(f"\n✅ Best iteration: {final_model.get_best_iteration()}")

# Evaluate
val_preds = final_model.predict(X_val_sel)
evaluate_predictions(y_val, val_preds, "Final CatBoost")

In [ ]:
# ============================================================================
# STEP 7: STACKING ENSEMBLE (Optional)
# ============================================================================

# Uncomment to build stacking ensemble
# stacker = build_stacking_ensemble(
#     X_train_sel, y_train, best_params, cv_folds=5, use_gpu=USE_GPU
# )
# stacker_preds = stacker.predict(X_val_sel)
# evaluate_predictions(y_val, stacker_preds, "Stacking Ensemble")

In [ ]:
# ============================================================================
# STEP 8: GENERATE SUBMISSION
# ============================================================================

# Generate predictions
test_predictions = final_model.predict(X_test_sel)

# Create submission
submission = create_submission(
    predictions=test_predictions,
    test_index=X_official_test.index,
    y_train=y_train,
    filepath='submission_final.csv'
)

print("\n🎉 Pipeline Complete!")

In [ ]:
# ============================================================================
# STEP 9: MODEL INTERPRETATION (Optional)
# ============================================================================

# Uncomment to run SHAP analysis
# shap_values, X_explain, explainer = explain_model(final_model, X_val_sel)
# plot_shap_summary(shap_values, X_explain)
# plot_shap_waterfall(shap_values, X_explain, explainer, sample_idx=0)

In [ ]:
# ============================================================================
# STEP 10: SAVE MODELS
# ============================================================================

# Save final model
save_model(final_model, 'catboost_final_model.cbm')

# Save parameters and features for reproducibility
save_params(best_params, 'best_hyperparameters.json')

with open('selected_features.json', 'w') as f:
    json.dump(selected_features, f, indent=4)
print("💾 Features saved to selected_features.json")

---
## 📝 Summary

This notebook provides a complete ML pipeline for student math score prediction:

1. **Memory-efficient data loading** with automatic type optimization
2. **Robust preprocessing** for both boosting and sklearn models
3. **Multiple model comparison** (LightGBM, XGBoost, CatBoost)
4. **Optuna hyperparameter optimization**
5. **Feature selection** via recursive elimination
6. **Advanced ensembling** (stacking, meta-learning)
7. **Distribution shift detection** via adversarial validation
8. **Neural network error correction**
9. **SHAP-based model interpretation**

### Key Features:
- Type hints for better code readability
- Comprehensive docstrings
- GPU support with automatic fallback
- Modular, reusable functions
- Progress tracking with tqdm